# Đánh giá chất lượng dữ liệu Yelp (báo cáo CLDL + TEMPO-DQ)

Notebook mỏng: logic ở package `dq/`, quy tắc ở `config.yaml`, định nghĩa độ đo ở `METRICS.md`. Đơn vị đo là **ô**; mỗi hàm kiểm tra trả True / False / N/A. Chạy từ trên xuống; mọi con số do code sinh ra.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd()
if not (ROOT / "dq").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from dq import ASPECTS
from dq.run import load_config, run

DATA_DIR = os.environ.get("DQ_DATA_DIR", r"C:\Users\Yoga\OneDrive\Desktop\UIT code\ds317\archive")
OUT_DIR  = os.environ.get("DQ_OUT_DIR", str(ROOT / "working_v0"))
CONFIG   = os.environ.get("DQ_CONFIG", str(ROOT / "config.yaml"))
LABEL    = os.environ.get("DQ_LABEL", "KQ_v0")      # KQ_v0 = dữ liệu gốc; KQ_v1 = sau cải thiện
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
pd.set_option("display.max_colwidth", 80)

## 1. Chạy đánh giá tự động

In [ ]:
cfg = load_config(CONFIG, data_dir=DATA_DIR, out_dir=OUT_DIR, label=LABEL)
result = run(cfg)          # giới hạn bảng: run(cfg, tables=["business", "review"])
frames = result["frames"]

## 2. Ma trận điểm khía cạnh (trung bình điểm các quy tắc xếp hạng; ⚑ = có quy tắc cứng ★ ở mức Tệ)

In [ ]:
a = frames["aspects"]
a = a[a["aspect"].isin(ASPECTS)].copy()
a["cell"] = a.apply(lambda r: ("N/A" if r["status"] == "N/A" else "Chỉ báo cáo" if r["status"] == "Chỉ báo cáo"
                               else f"{r['score']:.4f} {r['grade']}" + (" ⚑" if r["flag"] else "")), axis=1)
display(a.pivot(index="table", columns="aspect", values="cell")[ASPECTS])

Khía cạnh **không áp dụng / chỉ báo cáo** và lý do:

In [ ]:
display(a[a["status"] != "Đo được"][["table", "aspect", "status", "note"]].reset_index(drop=True))

## 3. Chi tiết từng quy tắc: số ô xét, số ô vi phạm, điểm, xếp hạng, hành động

In [ ]:
r = frames["rules"]
cols = ["table", "rule_id", "aspect", "level", "n_units", "n_violations", "score", "grade", "action"]
display(r[cols].sort_values(["table", "aspect", "rule_id"]).reset_index(drop=True))

## 4. Mức bộ dữ liệu (trung bình các bảng)

In [ ]:
display(frames["dataset"])

## 5. Quy tắc vi phạm nhiều nhất và giá trị lỗi phổ biến

In [ ]:
display(r[r["n_violations"] > 0].sort_values("n_violations", ascending=False).head(15)[["table", "rule_id", "n_violations", "score", "grade", "top_values"]])

Chỉ số chỉ báo cáo (C8 độ phủ, S6 tổng review_count so với số review quan sát, T4 tỷ lệ POI còn hoạt động):

In [ ]:
display(frames["metrics"])

## 6. Biểu đồ theo bảng

In [ ]:
for t in cfg["tables"]:
    p = Path(OUT_DIR) / f"chart_{t}.png"
    if p.exists():
        display(Image(filename=str(p)))

## 7. Dẫn chứng: ô lỗi mẫu (đủ khóa để tra lại dữ liệu gốc)

In [ ]:
smp = frames["samples"]
RULE = None   # ví dụ "U1"; None = 3 mẫu đầu của mọi quy tắc
view = smp[smp["rule_id"] == RULE] if RULE else smp.groupby(["table", "rule_id"]).head(3)
display(view.head(60))

## 8. Hồ sơ thời gian (thông tin) và đối chiếu TEMPO-DQ Eq.(2)

In [ ]:
display(frames["profile"].drop(columns=["by_year"]))
display(frames["eq2"][frames["eq2"]["n_missing"] > 0])

## 9. So sánh trước - sau (sau khi có KQ_v1)

`python -m dq.compare working_v0 working_v1 --out so_sanh.md`

## 10. Báo cáo tự sinh (`report.md`)

In [ ]:
display(Markdown(result["report_md"]))

## 11. Nhận định (viết tay sau khi xem số liệu)

> Chỉ viết dạng *"theo bảng …, quy tắc …"* và dẫn lại số từ các bảng trên; không gõ lại con số bằng tay.